In [24]:
import torch
import random
import numpy as np
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EvalPrediction,
)
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
from collections import defaultdict
from sklearn.metrics import roc_auc_score
from transformers import EarlyStoppingCallback

# ── 配置 ──────────────────────────────────────────────────────────────────────
MODEL_NAME = "/root/.cache/modelscope/hub/models/BAAI/bge-reranker-v2-m3"
TRAIN_GROUP_SIZE = 8
MAX_LEN = 512

LORA_CONFIG = LoraConfig(
    task_type=TaskType.SEQ_CLS,       # 序列分类任务
    r=16,                             # LoRA 秩，越大参数越多效果越好，显存也越大
    lora_alpha=32,                    # 缩放系数，通常设为 r 的 2 倍
    lora_dropout=0.1,
    bias="none",
    # bge-reranker-v2-m3 基于 XLM-RoBERTa，attention 的投影层名称如下
    target_modules=["query", "key", "value"],
)

In [2]:
# ── 1. 数据转换 ────────────────────────────────────────────────────────────────

def build_groups(raw_data, group_size=TRAIN_GROUP_SIZE, seed=42):
    """
    输入: [{'query': ..., 'passage': ..., 'label': 0/1}, ...]
    输出: [{'query': ..., 'positive': ..., 'negatives': [...]}, ...]
    
    策略：
    - 每个 (query, positive) 搭配一组负样本构成一个 group
    - 如果负样本不足 group_size-1，则有放回地重复采样
    """
    random.seed(seed)
    
    # 按 query 分组
    query_to_pos = defaultdict(list)
    query_to_neg = defaultdict(list)
    
    for item in raw_data:
        q = item["query"]
        if item["label"] == 1:
            query_to_pos[q].append(item["passage"])
        else:
            query_to_neg[q].append(item["passage"])
    
    groups = []
    n_neg_needed = group_size - 1
    
    for query, positives in query_to_pos.items():
        negatives = query_to_neg.get(query, [])
        
        if len(negatives) == 0:
            # 没有负样本则跳过（或可以从其他 query 的 passage 里借）
            print(f"[Warning] query 没有负样本，已跳过: {query[:30]}...")
            continue
        
        for positive in positives:
            if len(negatives) >= n_neg_needed:
                # 无放回采样，避免重复
                sampled_negs = random.sample(negatives, n_neg_needed)
            else:
                # 负样本不足时，有放回采样补齐
                sampled_negs = random.choices(negatives, k=n_neg_needed)
            
            groups.append({
                "query": query,
                "positive": positive,
                "negatives": sampled_negs,
            })
    
    random.shuffle(groups)
    return groups

In [3]:
class RerankerGroupDataset(Dataset):
    def __init__(self, groups, tokenizer, max_len=MAX_LEN):
        self.groups = groups
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.groups)

    def __getitem__(self, idx):
        g = self.groups[idx]
        passages = [g["positive"]] + g["negatives"]
        pairs = [[g["query"], p] for p in passages]
        encoded = self.tokenizer(
            pairs,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids": encoded["input_ids"],
            "attention_mask": encoded["attention_mask"],
        }

In [4]:
class RerankerEvalDataset(Dataset):
    """
    eval 不需要 group，每条样本是一个 (query, passage, label) 的 pair
    label 直接传给 compute_metrics 计算 AUC
    """
    def __init__(self, raw_data, tokenizer, max_len=MAX_LEN):
        self.data = raw_data
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        encoded = self.tokenizer(
            [[item["query"], item["passage"]]],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids": encoded["input_ids"].squeeze(0),      # (seq_len,)
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "labels": torch.tensor(item["label"], dtype=torch.float),
        }

In [10]:
class RerankerWithLoRA(torch.nn.Module):
    def __init__(self, model_name, lora_config, group_size=TRAIN_GROUP_SIZE):
        super().__init__()
        base_model = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=1
        )
        self.model = get_peft_model(base_model, lora_config)
        self.model.enable_input_require_grads()
        self.model.print_trainable_parameters()
        self.group_size = group_size

    # ── 新增：代理 gradient checkpointing 相关方法 ──
    def gradient_checkpointing_enable(self, **kwargs):
        self.model.gradient_checkpointing_enable(**kwargs)

    def gradient_checkpointing_disable(self):
        self.model.gradient_checkpointing_disable()

    def forward(self, input_ids, attention_mask, labels=None):
        # ── 训练路径：input_ids shape = (batch, group_size, seq_len) ──
        if input_ids.dim() == 3:
            batch, gs, seq_len = input_ids.shape
            scores = self.model(
                input_ids=input_ids.view(batch * gs, seq_len),
                attention_mask=attention_mask.view(batch * gs, seq_len),
            ).logits.squeeze(-1)

            scores = scores.view(batch, gs)
            targets = torch.zeros(batch, dtype=torch.long, device=scores.device)
            loss = torch.nn.functional.cross_entropy(scores, targets)
            return {"loss": loss, "logits": scores}

        # ── Eval 路径：input_ids shape = (batch, seq_len) ──
        else:
            output = self.model(
                input_ids=input_ids,
                attention_mask=attention_mask,
            )
            scores = output.logits.squeeze(-1)   # (batch,)

            loss = None
            if labels is not None:
                # eval 时用 BCE 算一个 loss（可选，主要看 AUC）
                loss = torch.nn.functional.binary_cross_entropy_with_logits(
                    scores, labels.float()
                )
            return {"loss": loss, "logits": scores}

In [11]:
def compute_metrics(eval_pred: EvalPrediction):
    logits, labels = eval_pred
    # logits shape: (n_samples,) 或 (n_samples, 1)
    scores = logits.squeeze(-1) if logits.ndim == 2 else logits
    scores = 1 / (1 + np.exp(-scores))   # sigmoid，转成概率

    auc = roc_auc_score(labels.astype(int), scores)
    return {"auc": auc}

In [25]:
def train(train_data, eval_data):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    groups = build_groups(train_data)
    print(f"共构建 {len(groups)} 个训练 groups")

    train_dataset = RerankerGroupDataset(groups, tokenizer)
    eval_dataset = RerankerEvalDataset(eval_data, tokenizer)
    model = RerankerWithLoRA(MODEL_NAME, LORA_CONFIG)

    training_args = TrainingArguments(
        output_dir="./reranker-lora",
        num_train_epochs=1,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        gradient_checkpointing=True,
        learning_rate=2e-4,
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        fp16=True,
        # ── eval 相关 ──────────────────────────────────
        eval_strategy="steps",
        eval_steps=50,                 # 每 500 步 eval 一次
        per_device_eval_batch_size=32,   # eval 不需要 group，batch 可以开大
        save_strategy="steps",
        save_steps=500,
        load_best_model_at_end=True,     # 训练结束后自动加载 AUC 最高的 checkpoint
        metric_for_best_model="auc",
        greater_is_better=True,
        # ───────────────────────────────────────────────
        logging_steps=50,
        report_to="none",
        optim="adamw_8bit",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        compute_metrics=compute_metrics,
        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=3,   # 连续 3 次 eval AUC 没有提升则停止
                early_stopping_threshold=0.001,  # AUC 提升小于 0.001 视为没有提升
            )
        ],
    )
    trainer.train()

    model.model.save_pretrained("../ft_data/group-style-reranker-lora/adapter")
    tokenizer.save_pretrained("../ft_data/group-style-reranker-lora/adapter")
    print("训练完成，最优 adapter 已保存。")

In [26]:
class Reranker:
    def __init__(self, base_model_name, adapter_path, device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(adapter_path)
        base_model = AutoModelForSequenceClassification.from_pretrained(
            base_model_name, num_labels=1
        )
        self.model = PeftModel.from_pretrained(base_model, adapter_path).to(self.device)
        self.model.eval()

    @torch.no_grad()
    def rerank(self, query, passages, batch_size=32, normalize=True):
        pairs = [[query, p] for p in passages]
        all_scores = []

        for i in range(0, len(pairs), batch_size):
            batch_pairs = pairs[i: i + batch_size]
            encoded = self.tokenizer(
                batch_pairs,
                max_length=MAX_LEN,
                padding=True,
                truncation=True,
                return_tensors="pt",
            ).to(self.device)

            scores = self.model(**encoded).logits.squeeze(-1)
            if normalize:
                scores = torch.sigmoid(scores)
            all_scores.extend(scores.cpu().tolist())

        return sorted(zip(passages, all_scores), key=lambda x: x[1], reverse=True)


In [27]:
import json

train_raw_data = []

with open("../ft_data/train_without_rand.jsonl", encoding='utf-8') as inf:
    for line in inf:
        d = json.loads(line.strip())
        if d['label'] == "1":
            d['label'] = 1
        elif d['label'] == "0":
            d['label'] = 0
        train_raw_data.append(d)

valid_raw_data = []

with open("../ft_data/valid.jsonl", encoding='utf-8') as inf:
    for line in inf:
        d = json.loads(line.strip())
        if d['label'] == "1":
            d['label'] = 1
        elif d['label'] == "0":
            d['label'] = 0
        valid_raw_data.append(d)

train(train_raw_data, valid_raw_data)

[Warning] query 没有负样本，已跳过: Pfändung Grundstück Gasometers...
[Warning] query 没有负样本，已跳过: Stiftung C. – Rekurs gegen Bau...
[Warning] query 没有负样本，已跳过: Unfallversicherung § 8 SGB VII...
[Warning] query 没有负样本，已跳过: Notar – Prüfungspflichten – Vo...
[Warning] query 没有负样本，已跳过: Vorstrafen Einfluss Strafzumes...
[Warning] query 没有负样本，已跳过: Steuerbehörde Verkehrswert Anl...
[Warning] query 没有负样本，已跳过: Verwaltungsverfügung Gebühr Er...
[Warning] query 没有负样本，已跳过: „Hacking“ OR „unerlaubter Zugr...
[Warning] query 没有负样本，已跳过: Stiftungserbrecht · Schweiz · ...
[Warning] query 没有负样本，已跳过: Zürich, Baurekursgericht, Denk...
[Warning] query 没有负样本，已跳过: Haftung für Schäden bei Besuch...
[Warning] query 没有负样本，已跳过: „Kaufvertrag Immobilie Mängel ...
共构建 13902 个训练 groups
trainable params: 3,409,921 || all params: 571,165,698 || trainable%: 0.597010816990624


/root/miniconda3/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss,Auc
50,1.692900,0.474406,0.782962
100,1.400000,0.528705,0.794272
150,1.281400,0.440473,0.821592
200,1.199800,0.521423,0.832874
250,1.114800,0.517193,0.848463
300,1.037800,0.443403,0.853603
350,1.074500,0.559748,0.856869
400,0.941300,0.430528,0.868376
450,0.978500,0.453444,0.864485
500,0.963400,0.692126,0.862840


/root/miniconda3/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/root/miniconda3/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/root/miniconda3/lib/python3.12/site-packa

训练完成，最优 adapter 已保存。
